In [ ]:
# Cell 1 - Setup
from dotenv import load_dotenv
from groq import Groq
import base64
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = "llama-3.3-70b-versatile"

In [ ]:
# Cell 2 - Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop=None):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages += messages

    response = client.chat.completions.create(
        model=model,
        max_tokens=4000,
        messages=all_messages,
        temperature=temperature,
        stop=stop,
    )
    return response

def text_from_message(response):
    return response.choices[0].message.content or ""

In [ ]:
# Cell 3 - Article text
article_text = """
Earth is the third planet from the Sun and the only
astronomical object known to harbor life. This is
enabled by Earth being an ocean world, the only one in
the Solar System sustaining liquid surface water. Almost
all of Earth's water is contained in its global ocean,
covering 70.8% of Earth's crust. The remaining 29.2% of
Earth's crust is land, most of which is located in the
form of continental landmasses within Earth's land
hemisphere. Most of Earth's land is at least somewhat
humid and covered by vegetation, while large sheets of
ice at Earth's polar deserts retain more water than
Earth's groundwater, lakes, rivers, and atmospheric
water combined. Earth's crust consists of slowly moving
tectonic plates, which interact to produce mountain
ranges, volcanoes, and earthquakes. Earth has a liquid
outer core that generates a magnetosphere capable of
deflecting most of the destructive solar winds and
cosmic radiation.
Earth has a dynamic atmosphere, which sustains
Earth's surface conditions and protects it from most
meteoroids and UV-light at entry. It has a composition
of primarily nitrogen and oxygen. Water vapor is widely
present in the atmosphere, forming clouds that cover
most of the planet. The water vapor acts as a greenhouse
gas and, together with other greenhouse gases in the
atmosphere, particularly carbon dioxide (CO2), creates
the conditions for both liquid surface water and water
vapor to persist via the capturing of energy from the
Sun's light. This process maintains the current average
surface temperature of 14.76 °C (58.57 °F), at which
water is liquid under normal atmospheric pressure.
Earth's atmosphere and oceans were formed by volcanic activity and outgassing.
Water vapor from these sources condensed into the oceans, augmented by water 
and ice from asteroids, protoplanets, and comets.
Sufficient water to fill the oceans may have been on Earth since it formed.
In this model, atmospheric greenhouse gases kept the oceans from freezing when 
the newly forming Sun had only 70% of its current luminosity.
By 3.5 Ga, Earth's magnetic field was established, which helped prevent the 
atmosphere from being stripped away by the solar wind.
"""

In [ ]:
# Cell 4 - Citation system prompt (simulates Anthropic's citations feature)
citation_system = """You are a research assistant that answers questions using ONLY 
the provided document. 

When answering:
1. Quote the exact relevant sentence(s) from the document using quotation marks
2. After each quote write [Quote X] where X is a number
3. At the end list all quotes under "## Sources" like:
   [Quote 1]: "exact text from document"
   [Quote 2]: "exact text from document"
4. Never add information not present in the document
5. If the document doesn't contain the answer, say so clearly"""

In [ ]:
# Cell 5 - Query with citations (text document)
messages = []

prompt = f"""Here is the document to reference:

<document title="Earth Article">
{article_text}
</document>

Question: How were Earth's atmosphere and oceans formed?"""

add_user_message(messages, prompt)

response = chat(messages, system=citation_system)
print(text_from_message(response))

In [ ]:
# Cell 6 - Query with PDF file (read as text)
def read_pdf_as_text(pdf_path):
    """Extract text from PDF using PyMuPDF (free)"""
    try:
        import fitz  # pip install pymupdf
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except ImportError:
        print("Install pymupdf: pip install pymupdf")
        return None

# Use PDF if available, otherwise use article_text
pdf_path = "earth.pdf"
if os.path.exists(pdf_path):
    pdf_text = read_pdf_as_text(pdf_path)
    document_text = pdf_text if pdf_text else article_text
    print("✅ Using PDF")
else:
    document_text = article_text
    print("✅ Using article text")

messages = []
prompt = f"""Here is the document to reference:

<document title="Earth Article">
{document_text}
</document>

Question: How were Earth's atmosphere and oceans formed?"""

add_user_message(messages, prompt)
response = chat(messages, system=citation_system)
print(text_from_message(response))

In [ ]:
# Cell 7 - Multi-turn with citations
messages = []

# First question
prompt = f"""<document title="Earth Article">
{article_text}
</document>

Question: What percentage of Earth's surface is covered by ocean?"""

add_user_message(messages, prompt)
response = chat(messages, system=citation_system)
answer1 = text_from_message(response)
print("Q1 Answer:")
print(answer1)

# Add to history
add_assistant_message(messages, answer1)

# Follow-up question
add_user_message(messages, "What is the average surface temperature mentioned?")
response2 = chat(messages, system=citation_system)
print("\nQ2 Answer:")
print(text_from_message(response2))